Unscrambling the files that contain the names of each gfx addon.

In [ ]:
import glob
from itertools import batched
from pathlib import Path
from warnings import warn

from PIL import Image
from IPython.display import display

In [ ]:
BASEDIR = Path("./tilesets/")

In [ ]:
for f in sorted(BASEDIR.glob("_jetp_?.dat", case_sensitive=False), key=lambda f: f.name.lower()):
    data = f.read_bytes()
    newdata = bytes(b ^ 52 for b in data)
    print("{:>12} {}".format(f.name, newdata.decode('ascii')))

In [ ]:
bin(52)

In [ ]:
hex(52)

---

Now exploring something else

In [ ]:
for f in BASEDIR.glob("_jetp_?0.dat", case_sensitive=False):
    data = f.read_bytes()
    newdata = bytes(b ^ 48 for b in data)
    print(newdata[:48])

---

I believe `JETDEMO.DAT` is simple:
- First byte is the level to be loaded, zero-indexed. `0x26` = `38` which loads level 39 ("Logo").
- Each of the following bytes is the input. One byte per frame, one bit per input. I guess:
    - bit 0 (`0x01`) is right
    - bit 1 (`0x02`) is left
    - bit 2 (`0x04`) is down
    - bit 3 (`0x08`) is up
    - bit 4 (`0x10`) is jump
    - bit 5 (`0x20`) is phaser
This has not been confirmed yet.

I could probably post this research to https://tcrf.net/ and/or https://datacrystal.tcrf.net/
But first I'll write everything down and save on GitHub.

---

In [ ]:
class BitGenerator:
    def __init__(self, data: bytes):
        self.data = data
        self.pointer = 0
        self.buffer = []

    def remaining_bits(self):
        return len(self.buffer) + len(self.data) - self.pointer

    def is_empty(self):
        return self.pointer >= len(self.data) and len(self.buffer) == 0

    def consume_byte(self):
        out = self.data[self.pointer]
        self.pointer += 1
        return out

    def get_bit(self):
        if len(self.buffer) == 0:
            self.buffer.extend(int(bit) for bit in '{:08b}'.format(self.consume_byte()))
        return self.buffer.pop(0)

    def get_bits(self, how_many : int):
        out = 0
        for i in range(how_many):
            out <<= 1
            out |= self.get_bit()
        return out

    def get_byte(self):
        return self.get_bits(8)

In [ ]:
def gfxdat_parser(rawdata: bytes):
    # Raw palette has one byte per channel (R, G, B).
    # Each channel is 6-bit (0..63), as per VGA palette limitation.
    rawpalette = rawdata[:256*3]
    # Compressed stream of pixels.
    rawpixels = rawdata[256*3:]

    # One byte per channel (R, G, B), 8-bit per channel.
    palette = bytearray(round(c * 255 / 63) for c in rawpalette)

    # Our framebuffer where the compressed image will be uncompresed.
    pixels = bytearray(320*200)
    pointer = 0

    stream = BitGenerator(rawpixels)
    while pointer < len(pixels):
        is_repeating = stream.get_bit()
        qty = 1
        if is_repeating:
            qty = 2 + stream.get_bits(5)
        colorindex = stream.get_byte()
        for i in range(qty):
            pixels[pointer] = colorindex
            pointer += 1

    if (remainder := stream.remaining_bits()) >= 8:
        print("Warning: {} trailing bytes ({} bits) are being ignored after the pixel data.".format(remainder // 8, remainder))

    img = Image.new(mode="P", size=(320, 200))
    img.putpalette(palette)
    img.putdata(pixels)

    return img

In [ ]:
# XXX: This is a hacky experiment! NOT ready yet!
# The xjetpack uses a different format for the graphics.
# I don't know this format yet.
# This code here is just a quick copy-and-paste, for quick experimentation.
def xmas_gfxdat_parser(rawdata: bytes):
    # Raw palette has one byte per channel (R, G, B).
    # Each channel is 6-bit (0..63), as per VGA palette limitation.
    #rawpalette = rawdata[:256*3]
    # Compressed stream of pixels.
    rawpixels = rawdata

    palette = default_palette

    # Our framebuffer where the compressed image will be uncompresed.
    pixels = bytearray(320*200)
    pointer = 0

    stream = BitGenerator(rawpixels)
    while pointer < len(pixels):
        is_repeating = stream.get_bit()
        qty = 1
        if is_repeating:
            qty = 2 + stream.get_bits(5)
        colorindex = stream.get_byte()
        for i in range(qty):
            pixels[pointer] = colorindex
            pointer += 1
            if pointer >= len(pixels):
                break

    if (remainder := stream.remaining_bits()) >= 8:
        print("Warning: {} trailing bytes ({} bits) are being ignored after the pixel data.".format(remainder // 8, remainder))

    img = Image.new(mode="P", size=(320, 200))
    img.putpalette(palette)
    img.putdata(pixels)

    return img

In [ ]:
default_palette = gfxdat_parser((BASEDIR / '_JETP_A0.DAT').read_bytes()).getpalette()

In [ ]:
for f in BASEDIR.glob("_jetp_?0.dat", case_sensitive=False):
    display(gfxdat_parser(f.read_bytes()))

In [ ]:
for filename in sorted(glob.iglob("/home/denilson/Games/*/JETPACK?.DAT")):
    print(filename)
    with open(filename, "rb") as f:
        if "xjetpack" in filename:
            img = xmas_gfxdat_parser(f.read())
        else:
            img = gfxdat_parser(f.read())
        display(img)